# Module 322 — Adversarial Training Components

This notebook implements the components introduced in the course section: LeNet-5, FGSM, I-FGSM, and evaluation helpers. Training is deliberately opt-in. The defaults use the CPU, one PyTorch thread, zero data-loader workers, and no pinned memory for stability on Windows.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Safe defaults for the learner's local environment.
USE_CUDA = False
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if USE_CUDA and torch.cuda.is_available() else "cpu")
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
EPSILON = 0.3
I_FGSM_STEPS = 10
RUN_TRAINING = False


## LeNet-5

The two convolution/pooling blocks turn a `1x28x28` grayscale image into `16x5x5` features, or 400 numbers. The final linear layer returns ten logits—one raw score per digit. We do not apply softmax here because cross-entropy loss expects logits directly.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = LeNet5().to(DEVICE)
assert model(torch.zeros(2, 1, 28, 28, device=DEVICE)).shape == (2, 10)


## FGSM: the formula in code

Read `x-adv = x + epsilon * sign(gradient-x L(f(x), y))` as **adversarial x equals x plus epsilon times the sign of the gradient of the loss with respect to x**. `clone().detach()` protects the caller's tensor and starts a fresh computation graph; `requires_grad_(True)` asks autograd for the pixel gradient. Clamping uses normalized MNIST bounds, not raw `[0,1]` bounds.

In [ ]:
def fgsm_attack(model, images, labels, epsilon=EPSILON):
    images_copy = images.clone().detach().requires_grad_(True)
    outputs = model(images_copy)
    loss = F.cross_entropy(outputs, labels)
    model.zero_grad(set_to_none=True)
    loss.backward()
    delta = epsilon * images_copy.grad.sign()  # implements epsilon * sign(gradient)
    adv_images = images_copy + delta
    low, high = (0 - MNIST_MEAN) / MNIST_STD, (1 - MNIST_MEAN) / MNIST_STD
    return torch.clamp(adv_images, low, high).detach()


## I-FGSM: repeated steps plus projection

I-FGSM uses `alpha = epsilon / steps`, recomputes the gradient at each look-ahead point, and projects the cumulative perturbation back into `[-epsilon, epsilon]`. Projection is what keeps ten small moves from exceeding the original budget.

In [ ]:
def i_fgsm_attack(model, images, labels, epsilon=EPSILON, steps=I_FGSM_STEPS):
    alpha = epsilon / steps
    low, high = (0 - MNIST_MEAN) / MNIST_STD, (1 - MNIST_MEAN) / MNIST_STD
    original = images.clone().detach()
    adv = original.clone()
    for _ in range(steps):
        adv.requires_grad_(True)
        loss = F.cross_entropy(model(adv), labels)
        model.zero_grad(set_to_none=True)
        loss.backward()
        adv = (adv.detach() + alpha * adv.grad.sign())
        perturbation = torch.clamp(adv - original, -epsilon, epsilon)
        adv = torch.clamp(original + perturbation, low, high)
    return adv.detach()


Keep evaluation separate from training: attacks need gradients, while final predictions should run under `torch.no_grad()`. For large runs, evaluate a bounded number of batches first; this is both faster and safer for memory.

In [ ]:
def evaluate_adversarial_accuracy(model, loader, epsilon, num_batches=None):
    model.eval()
    correct = total = 0
    for i, (images, labels) in enumerate(loader):
        if num_batches is not None and i >= num_batches:
            break
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        adv_images = fgsm_attack(model, images, labels, epsilon)
        with torch.no_grad():
            predicted = model(adv_images).argmax(dim=1)
        total += labels.numel()
        correct += (predicted == labels).sum().item()
    return 100.0 * correct / total if total else 0.0

if RUN_TRAINING:
    raise NotImplementedError("Add the course data loader and opt-in training loop after reviewing resource limits.")
else:
    print("Component notebook validated; training is disabled by default.")


## Opt-in epsilon-spread training skeleton

The course generates attacks in bounded batches and moves each result to CPU. The training function below is intentionally a skeleton: connect it to the course loader only after checking available memory.

In [ ]:
EPSILON_SPREAD = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

def train_one_batch(model, images, labels, optimizer, epsilon):
    # Current-model attack approximates the inner max in the min-max objective.
    adv_images = fgsm_attack(model, images, labels, epsilon)
    combined_images = torch.cat([images, adv_images], dim=0)
    combined_labels = torch.cat([labels, labels], dim=0)
    permutation = torch.randperm(combined_images.size(0), device=combined_images.device)
    optimizer.zero_grad(set_to_none=True)
    outputs = model(combined_images[permutation])
    loss = F.cross_entropy(outputs, combined_labels[permutation])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.detach()

if RUN_TRAINING:
    print("Run a bounded, monitored training loop here.")
